# Distances and ordination

**R equivalent:** `distance()`, `UniFrac()`, `ordinate()`, `plot_ordination()`

This notebook reproduces the classic phyloseq ordination workflow on the
GlobalPatterns dataset.

In [ ]:
import numpy as np
import pyloseq
from pyloseq import (
    Phyloseq, OtuTable, SampleData, TaxTable, PhyTree,
    distance, distance_method_list, ordinate, plot_ordination,
)
from pyloseq.testing.fixtures import load_global_patterns_reference

ref = load_global_patterns_reference()
gp = Phyloseq(
    otu=OtuTable(ref["otu_table"], taxa_are_rows=True),
    sam=SampleData(ref["sample_data"]),
    tax=TaxTable(ref["tax_table"]),
    tree=PhyTree.from_newick(ref["phy_tree_newick"]),
)
print(gp)

## Available distance methods

In [ ]:
methods = distance_method_list()
for group, names in methods.items():
    print(f"{group}: {names}")

## Bray-Curtis dissimilarity

**R equivalent:** `distance(gp, "bray")`

In [ ]:
dm_bray = gp.distance("bray")
print(f"Distance matrix: {len(dm_bray.ids)} × {len(dm_bray.ids)}")
print(f"Min non-zero: {np.array(dm_bray.data)[np.array(dm_bray.data) > 0].min():.4f}")
print(f"Max:          {np.array(dm_bray.data).max():.4f}")

# Sanity: zero diagonal, symmetric
arr = np.array(dm_bray.data)
assert np.allclose(np.diag(arr), 0), "Diagonal must be zero"
assert np.allclose(arr, arr.T), "Distance matrix must be symmetric"

## PCoA on Bray-Curtis

**R equivalent:** `ordinate(gp, "PCoA", "bray")`

In [ ]:
pcoa_result = gp.ordinate("PCoA", distance="bray")
prop_exp = pcoa_result.proportion_explained
print(f"Axis 1 explains {prop_exp.iloc[0]*100:.1f}% of variance")
print(f"Axis 2 explains {prop_exp.iloc[1]*100:.1f}% of variance")

# R reference: Axis.1 ~28%, Axis.2 ~16% on GlobalPatterns Bray-Curtis
assert prop_exp.iloc[0] > 0.15, "Axis 1 should explain > 15% of variance"
assert prop_exp.iloc[1] > 0.05, "Axis 2 should explain > 5% of variance"

## Plot the ordination

**R equivalent:** `plot_ordination(gp, pcoa_result, color="SampleType")`

In [ ]:
p = plot_ordination(gp, pcoa_result, color="SampleType", title="PCoA — Bray-Curtis")
print(p)

## PERMANOVA one-liner

Distance matrices are `skbio.DistanceMatrix` objects — plug directly into
`skbio.stats.distance.permanova` without any conversion.

**R equivalent:** `vegan::adonis2(dist ~ SampleType, data=sample_data(gp))`

In [ ]:
from skbio.stats.distance import permanova

grouping = gp.sample_data.to_frame()["SampleType"]
result = permanova(gp.distance("bray"), grouping, permutations=99)
print(result)
assert result["p-value"] is not None, "permanova should return a p-value"

## Unweighted UniFrac

**R equivalent:** `UniFrac(gp, weighted=FALSE)`

In [ ]:
from pyloseq import unifrac

dm_uf = unifrac(gp, weighted=False)
print(f"UniFrac matrix: {len(dm_uf.ids)} × {len(dm_uf.ids)}")

# UniFrac values are in [0, 1]
arr_uf = np.array(dm_uf.data)
assert arr_uf.min() >= 0
assert arr_uf.max() <= 1
print(f"Range: [{arr_uf[arr_uf > 0].min():.4f}, {arr_uf.max():.4f}]")